# Prairie Vigie – notebook 4 : où agir d'abord ? Priorités, tests de l'apport du modèle, données de la carte
À lancer **après les notebooks 1, 2 et 3, dans le même dossier** (cache et `sorties_prairies/` réutilisés).

**Principe** :
1. **Risque** : probabilité de conversion lors de la campagne 2027 (modèle retenu dans le notebook 3, régression logistique,
   lift 2,5 sur 2025). Pour comparer les territoires entre eux, l'historique observé fait mieux (test 5 bis).
2. **Enjeux** : pour chaque prairie, des drapeaux oui/non documentés : zone humide, aire d'alimentation de captage,
   habitat Natura 2000 (Calvados, Manche, Orne seulement : couverture de la carte), contour bordé de haies.
3. **Priorité** = parmi les 10 % de prairies les plus à risque, celles qui portent au moins un enjeu eau/biodiversité
   (zone humide, captage, Natura 2000) = niveau 1 ; et un second enjeu, haies comprises = niveau 2.
   (Révisé après le test de sensibilité : les haies seules, présentes autour de 60 % des prairies de l'ouest, ne sont pas sélectives.)
4. **Test de sensibilité** : on retire chaque enjeu à tour de rôle et on mesure si la liste des secteurs prioritaires change.
5. **Valoriser** : communes où les prairies se maintiennent (créations ≥ conversions présumées), observatoire 2019-2025.

In [ ]:
# 0. Sur Google Colab, le dossier DOSSIER est créé (ou repris) dans « Mon Drive » .
#    Les 4 notebooks doivent utiliser le même dossier.
DOSSIER = "datathon_prairies"
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")                                   # point de montage standard de Colab
    os.makedirs(os.path.join("/content/drive/MyDrive", DOSSIER), exist_ok=True)
    os.chdir(os.path.join("/content/drive/MyDrive", DOSSIER))
except ImportError:
    pass                                                            # hors Colab : dossier courant
print("Dossier de travail :", os.getcwd())
if not os.path.exists("sorties_prairies"):
    print("⚠ Dossier vide : lancer d'abord les notebooks précédents dans ce même dossier.")

In [ ]:
# 0 bis. Bibliothèques
!pip -q install geopandas pyogrio pyarrow scikit-learn

In [ ]:
# 1. Paramètres
import time
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, requests
from shapely.geometry import shape, box
from shapely.validation import make_valid
from shapely.ops import unary_union

DEPS = ["14", "27", "50", "61", "76"]
WFS = "https://data.geopf.fr/wfs/ows"
PAGE, DALLE = 5000, 20_000
CACHE = Path("cache_rpg"); CACHE.mkdir(exist_ok=True)
SORTIES = Path("sorties_prairies"); SORTIES.mkdir(exist_ok=True)
SESSION = requests.Session()
MIN_PP_MAILLE = 5                            # pas de diffusion d'une maille de 1 km avec moins de 5 prairies

PP, HERBE, GEL = 18, {17, 18, 19}, {11, 12, 13}
ARABLE = {1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 24, 25, 26}
def classe(g):
    g = int(g) if pd.notna(g) else -1
    return ("pp" if g == PP else "herbe" if g in HERBE else "arable" if g in ARABLE
            else "gel" if g in GEL else "autre")

def polygone(g):
    """Géométrie valide, parties surfaciques seulement."""
    g = make_valid(g)
    if g.geom_type == "GeometryCollection":
        parts = [p for p in g.geoms if p.geom_type in ("Polygon", "MultiPolygon")]
        g = unary_union(parts) if parts else None
    return g
import zipfile
CACHE_E = Path("cache_enjeux"); CACHE_E.mkdir(exist_ok=True)
AVEC_HAIES = True

In [ ]:
# 1. Risque 2027 par prairie.
V5 = SORTIES / "risque_2027_calibre_PRIVE.parquet"
TAB = pd.concat([pd.read_parquet(SORTIES / f"variables_{d}.parquet") for d in DEPS], ignore_index=True)
CODES = list(TAB["code_cultu"].value_counts().index[:8])
def codes_onehot(df):
    for c in CODES:
        df[f"code_{c}"] = (df["code_cultu"] == c).astype(int)
    return df
TAB = codes_onehot(TAB)
NON_VARS = {"pid", "y", "y2", "devenir", "transition", "code_cultu", "dep", "a0", "cx1", "cy1", "commune"}
VARS = [c for c in TAB.columns if c not in NON_VARS]
NOUV = codes_onehot(pd.concat([pd.read_parquet(SORTIES / f"v4_{d}.parquet") for d in DEPS], ignore_index=True))
base = pd.concat([TAB[TAB.a0 < 2022], NOUV], ignore_index=True)
base = base[base["y2"].notna()]
R = codes_onehot(pd.concat([pd.read_parquet(SORTIES / f"score_{d}_2025.parquet") for d in DEPS], ignore_index=True))
for v in VARS:
    if v not in R:
        R[v] = np.nan
# Score de CLASSEMENT : même modèle que le notebook 3 (régression logistique, entraînée sur a0 ≤ 2022), probabilité brute continue.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
tr = base[base.a0 <= 2022]
cols = [v for v in VARS if tr[v].notna().any()]
modele = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=3000)).fit(tr[cols], tr["y2"])
R["score"] = modele.predict_proba(R[cols])[:, 1]
if V5.exists():   # probabilité calibrée du notebook 3 : sert aux hectares attendus, pas au classement
    R = R.merge(pd.read_parquet(V5)[["dep", "pid", "p"]].rename(columns={"p": "risque"}), on=["dep", "pid"], how="left")
    print("Probabilités calibrées reprises du notebook 3 ; classement sur le score continu du même modèle")
else:
    R["risque"] = R["score"]
    print("Notebook 3 non lancé : risque = score brut (non calibré)")
R["top10"] = (R["score"].rank(method="first", ascending=False) <= 0.10 * len(R)).astype(int)
print(f"Contrôle : part des prairies dans le top 10 % = {R.top10.mean():.1%} ;",
      f"ex aequo évités : {R.risque.round(6).nunique():,} valeurs calibrées distinctes contre {R.score.nunique():,} scores")
R[["dep", "pid", "commune", "cx1", "cy1", "surface_ha", "ancienne", "score", "risque", "top10"]] \
  .to_parquet(SORTIES / "risque_parcelles_2027_PRIVE.parquet")       # privé
print(f"{len(R):,} prairies notées")

In [ ]:
# 2. Couches d'enjeux depuis data.gouv.fr (API officielle) : on affiche les ressources, on télécharge, on vérifie
def ressources(slug):
    r = SESSION.get(f"https://www.data.gouv.fr/api/1/datasets/{slug}/", timeout=120)
    r.raise_for_status()
    d = r.json()
    print(f"\n[{d.get('title')}] licence={d.get('license')} ; mise à jour={(d.get('last_update') or '')[:10]}")
    for x in d.get("resources", []):
        print(f"   - {x.get('format')!s:10} {x.get('title')!s:60.60} {x.get('url')}")
    return d.get("resources", [])

def candidats(slug):
    # Ressources réellement géographiques, de la plus fiable à la moins fiable (les gabarits vides sont exclus)
    out = []
    for i, x in enumerate(ressources(slug)):
        fmt, url, titre = str(x.get("format") or "").lower(), x.get("url") or "", str(x.get("title") or "").lower()
        u = url.lower()
        if any(m in titre for m in ("template", "trame", "canevas", "gabarit")) or "getcapabilities" in u:
            continue
        if fmt in ("wms", "wfs", "pdf", "csv") or "outputformat=csv" in u or "outputformat=midmif" in u:
            continue
        if "outputformat=gpkg" in u:
            prio = 0
        elif "outputformat=shapezip" in u or "atomarchive/getresource" in u:
            prio = 1
        elif "shp" in fmt or "shapefile" in fmt or fmt in ("zip", "geojson", "gpkg") or u.split("?")[0].endswith((".zip", ".gpkg", ".geojson", ".shp")):
            prio = 2
        else:
            continue
        if "_fra" in u or "france" in titre:      # couverture nationale complète en priorité (on découpe ensuite)
            prio -= 0.5
        out.append((prio, i, url))
    return sorted(out)

def telecharger_valide(nom, slug, essais=4):
    # Essaie les ressources dans l'ordre ; garde la première qui contient des polygones en Normandie
    dossier = CACHE_E / f"v2_{slug[:55]}"; dossier.mkdir(exist_ok=True)
    for prio, i, url in candidats(slug)[:essais]:
        f = next(iter(sorted(dossier.glob(f"donnees_{i}.*"))), None)
        if f is None:
            try:
                r = SESSION.get(url, timeout=1200)
                r.raise_for_status()
                ext = ".zip" if r.content[:2] == b"PK" else (".gpkg" if r.content[:15] == b"SQLite format 3" else ".geojson")
                f = dossier / f"donnees_{i}{ext}"
                f.write_bytes(r.content)
                print(f"   téléchargé : {url[:110]} ({len(r.content) / 1e6:.1f} Mo)")
            except Exception as e:
                print(f"   échec {url[:110]} : {e}")
                continue
        g = lire([f])
        if g is not None and len(g):
            return g
        print(f"   -> aucun polygone en Normandie dans {f.name}, ressource suivante")
    print(f"   ⚠ {nom} : aucune ressource exploitable, enjeu ignoré (à signaler)")
    return None

def lire_un(chemin):
    # Seule la géométrie sert : on n'ouvre pas les attributs (certains textes sont mal encodés ou tronqués)
    for options in ({"columns": []}, {"columns": [], "encoding": "latin-1"}, {}):
        try:
            return gpd.read_file(chemin, **options)
        except Exception as e:
            erreur = e
    raise erreur

def lire(fichiers):
    # Lit des zip (shp/gpkg/geojson contenus) ou fichiers simples.
    couches = []
    for fichier in fichiers or []:
        try:
            if fichier.suffix == ".zip":
                noms = [n for n in zipfile.ZipFile(fichier).namelist() if n.lower().endswith((".shp", ".gpkg", ".geojson"))]
                for n in noms:
                    g = lire_un(f"zip://{fichier}!{n}")
                    couches.append(g)
                    print(f"   {n} : {len(g):,} objets ; colonnes {list(g.columns)[:12]}")
            else:
                g = lire_un(fichier)
                couches.append(g)
                print(f"   {fichier.name} : {len(g):,} objets ; colonnes {list(g.columns)[:12]}")
        except Exception as e:
            print(f"   lecture impossible {fichier.name} : {e}")
    couches = [c for c in couches if c.crs is not None and c.geom_type.isin(["Polygon", "MultiPolygon"]).any()]
    if not couches:
        return None
    g = gpd.GeoDataFrame(pd.concat([c.to_crs(2154)[["geometry"]] for c in couches], ignore_index=True), crs=2154)
    normandie = unary_union(list(CONTOURS.values()))
    g = g[g.geometry.notna() & g.intersects(normandie)]
    g["geometry"] = g.geometry.apply(polygone)
    return g[g.geometry.notna()]

ENJEUX = {
    "zone_humide": "zones-humides-normandie",
    "captage": "aires-dalimentation-de-captages-france-entiere-1",
    "natura2000_habitat": "carte-de-synthese-des-habitats-dans-les-sites-natura-2000-de-normandie-occidentale-14-50-61-version-20260325",
}
COMMUNES = gpd.GeoDataFrame(pd.concat([gpd.read_parquet(CACHE / f"communes_{d}.parquet").assign(dep=d) for d in DEPS],
                                      ignore_index=True), geometry="geometry", crs=2154)
CONTOURS = {d: make_valid(unary_union(COMMUNES.loc[COMMUNES.dep == d, "geometry"].values)) for d in DEPS}
COUCHES_E = {}
for nom, slug in ENJEUX.items():
    f = CACHE_E / f"{nom}.parquet"
    if f.exists():
        COUCHES_E[nom] = gpd.read_parquet(f)
    else:
        g = telecharger_valide(nom, slug)
        if g is not None and len(g):
            g[["geometry"]].to_parquet(f)
            COUCHES_E[nom] = g[["geometry"]]
    print(f"==> {nom} : {len(COUCHES_E.get(nom, [])):,} polygones en Normandie,",
          f"{unary_union(COUCHES_E[nom].geometry.values).area / 1e4:,.0f} ha (sans double compte)" if nom in COUCHES_E else "ABSENT")

In [ ]:
# 3. Haies (BD TOPO, IGN, Licence Ouverte) par dalles adaptatives - géométrie seulement
def get(params):
    for essai in range(6):
        try:
            r = SESSION.get(WFS, params=params, timeout=300)
            if r.status_code == 200:
                return r.json()
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5 * 2 ** essai)
    raise RuntimeError("WFS indisponible")

def dalle_lignes(couche, x0, y0, x1, y1, zone, sortie, vus):
    if not box(x0, y0, x1, y1).intersects(zone):
        return
    page = get({"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature", "TYPENAMES": couche,
                "SRSNAME": "EPSG:2154", "BBOX": f"{x0},{y0},{x1},{y1},EPSG:2154",
                "OUTPUTFORMAT": "application/json", "COUNT": PAGE})
    feats = page.get("features", [])
    m = page.get("numberMatched", page.get("totalFeatures"))
    if len(feats) >= PAGE or (isinstance(m, int) and m > len(feats)):
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        for a, b, c, d in [(x0, y0, xm, ym), (xm, y0, x1, ym), (x0, ym, xm, y1), (xm, ym, x1, y1)]:
            dalle_lignes(couche, a, b, c, d, zone, sortie, vus)
        return
    for f in feats:
        if f.get("id") not in vus and f.get("geometry"):
            vus.add(f.get("id"))
            sortie.append(shape(f["geometry"]))

def haies(dep):
    f = CACHE_E / f"haies_{dep}.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    zone = CONTOURS[dep]
    xmin, ymin, xmax, ymax = zone.bounds
    lignes, vus = [], set()
    for x in np.arange(xmin, xmax, DALLE):
        for y in np.arange(ymin, ymax, DALLE):
            dalle_lignes("BDTOPO_V3:haie", x, y, x + DALLE, y + DALLE, zone, lignes, vus)
    g = gpd.GeoDataFrame(geometry=lignes, crs=2154)
    g.to_parquet(f)
    return g

if AVEC_HAIES:
    for d in DEPS:
        t0 = time.time()
        h = haies(d)
        print(f"{d} : {len(h):,} haies, {h.length.sum() / 1000:,.0f} km ({time.time() - t0:.0f} s)")

In [ ]:
# 4. Enjeux de chaque prairie 2025 (drapeaux documentés, seuils affichés)
SEUIL_SURFACE = 0.25        # une prairie « est » en zone humide / habitat si au moins 25 % de sa surface y est
SEUIL_HAIES = 0.50          # contour bordé de haies sur au moins 50 % de sa longueur (haie à moins de 10 m)
lignes = []
for dep in DEPS:
    f = SORTIES / f"enjeux_v4_{dep}.parquet"   # cache des enjeux (4e version du calcul, avec Natura 2000)
    if f.exists():
        lignes.append(pd.read_parquet(f)); continue
    t0 = time.time()
    g = gpd.read_parquet(CACHE / f"rpg_{dep}_2025_v3.parquet")
    g["pid"] = np.arange(len(g))
    g["classe"] = g["code_group"].apply(classe)
    pp = g[(g["classe"] == "pp") & g["dans_dep"]][["pid", "geometry"]].reset_index(drop=True)
    pp["surf"] = pp.area
    E = pd.DataFrame({"pid": pp["pid"].values, "dep": dep})
    for nom in ["zone_humide", "natura2000_habitat"]:
        if nom in COUCHES_E:
            inter = gpd.overlay(pp[["pid", "geometry"]], COUCHES_E[nom], how="intersection", keep_geom_type=True)
            part = (inter.assign(a=inter.area).groupby("pid")["a"].sum() / pp.set_index("pid")["surf"]).reindex(pp["pid"]).fillna(0).clip(upper=1)
            E[f"part_{nom}"] = part.values
            E[nom] = (part.values >= SEUIL_SURFACE).astype(int)
    if "captage" in COUCHES_E:
        pts = gpd.GeoDataFrame({"pid": pp["pid"]}, geometry=pp.representative_point(), crs=2154)
        j = gpd.sjoin(pts, COUCHES_E["captage"], how="left", predicate="within")
        E["captage"] = j.groupby(level=0)["index_right"].apply(lambda s: int(s.notna().any())).reindex(pts.index).values
    if AVEC_HAIES:
        anneau = gpd.GeoDataFrame({"pid": pp["pid"], "perim": pp.length}, geometry=pp.boundary.buffer(10), crs=2154)
        inter = gpd.overlay(haies(dep), anneau[["pid", "geometry"]], how="intersection", keep_geom_type=True)
        lg = inter.assign(l=inter.length).groupby("pid")["l"].sum()
        # longueur de haie dans l'anneau de 10 m / périmètre (borné à 1)
        E["part_contour_haies"] = (lg.reindex(pp["pid"]).fillna(0).values / anneau["perim"].values).clip(0, 1)
        E["haies"] = (E["part_contour_haies"] >= SEUIL_HAIES).astype(int)
    E.to_parquet(f)
    lignes.append(E)
    print(f"{dep} : {len(E):,} prairies ({time.time() - t0:.0f} s)")
EN = pd.concat(lignes, ignore_index=True)
DRAPEAUX = [c for c in ["zone_humide", "captage", "natura2000_habitat", "haies"] if c in EN]
print("Enjeux disponibles :", DRAPEAUX)
print("Part des prairies concernées par chaque enjeu, par département :")
print(EN.groupby("dep")[DRAPEAUX].mean().round(3).to_string())

In [ ]:
# 5. Priorité = risque élevé ET enjeu ; test de sensibilité ; sorties agrégées
P = R.merge(EN, on=["dep", "pid"], how="left")
P[DRAPEAUX] = P[DRAPEAUX].fillna(0).astype(int)
P["n_enjeux"] = P[DRAPEAUX].sum(axis=1)
# Révision documentée : au premier passage, retirer les haies changeait 48 % des secteurs prioritaires,
# car 60 % des prairies de l'ouest bocager ont un contour bordé de haies : ce critère seul n'est pas sélectif.
# Niveau 1 = risque élevé ET au moins un enjeu eau/biodiversité (zone humide, captage, habitat Natura 2000).
# Niveau 2 = niveau 1 ET au moins un second enjeu (dont les haies).
EAU_BIO = [d for d in ["zone_humide", "captage", "natura2000_habitat"] if d in DRAPEAUX]
P["prio1"] = ((P.top10 == 1) & (P[EAU_BIO].sum(axis=1) >= 1)).astype(int)
P["prio2"] = ((P.prio1 == 1) & (P.n_enjeux >= 2)).astype(int)
print(f"Prairies prioritaires niveau 1 : {P.prio1.sum():,} ({P.loc[P.prio1 == 1, 'surface_ha'].sum():,.0f} ha)")
print(f"Prairies prioritaires niveau 2 : {P.prio2.sum():,} ({P.loc[P.prio2 == 1, 'surface_ha'].sum():,.0f} ha)")
# Enregistré (nombre de prairies et hectares, Normandie) : ces chiffres sont cités dans le descriptif et le pitch
pd.DataFrame([{"niveau": n, "n_prairies": int(P[f"prio{n}"].sum()),
               "ha": round(P.loc[P[f"prio{n}"] == 1, "surface_ha"].sum(), 1)} for n in (1, 2)]) \
  .to_csv(SORTIES / "priorites_synthese_normandie.csv", index=False)
print("Répartition du nombre d'enjeux parmi les 10 % les plus à risque :",
      P.loc[P.top10 == 1, "n_enjeux"].value_counts().sort_index().to_dict())

# Sensibilité : en retirant un enjeu, les 50 mailles de 10 km les plus prioritaires changent-elles ?
# Échelles retenues : mailles de 10 km et intercos (corrélation prévu/observé ≥ 0,4, voir test 5 bis).
# La prédiction n'est pas publiée à la commune.
P["m10"] = (P.cx1 // 10).astype(int).astype(str) + "_" + (P.cy1 // 10).astype(int).astype(str)
def top_mailles(df, col, n=50):
    s = df[df[col] == 1].groupby("m10")["surface_ha"].sum()
    return set(s.sort_values(ascending=False).head(n).index)
ref = top_mailles(P, "prio1")
for d in EAU_BIO:
    autres = [x for x in EAU_BIO if x != d]
    Q = P.assign(p=((P.top10 == 1) & (P[autres].sum(axis=1) >= 1)).astype(int))
    print(f"   sans « {d} » : {len(ref & top_mailles(Q, 'p')) / max(len(ref), 1):.0%} des 50 mailles de 10 km prioritaires inchangées")

P["ha"] = P["surface_ha"]
def agreger(df, cles):
    a = df.groupby(cles, dropna=False).agg(n_pp=("pid", "size"), ha_pp=("ha", "sum"), risque_moyen=("risque", "mean"),
            ha_top10=("ha", lambda s: s[df.loc[s.index, "top10"] == 1].sum()),
            ha_prio1=("ha", lambda s: s[df.loc[s.index, "prio1"] == 1].sum()),
            ha_prio2=("ha", lambda s: s[df.loc[s.index, "prio2"] == 1].sum()),
            **{f"ha_{d}": ("ha", (lambda dd: (lambda s: s[df.loc[s.index, dd] == 1].sum()))(d)) for d in DRAPEAUX},
            **{f"ha_top10_{d}": ("ha", (lambda dd: (lambda s: s[(df.loc[s.index, dd] == 1) & (df.loc[s.index, "top10"] == 1)].sum()))(d)) for d in DRAPEAUX})
    return a.round(2).reset_index()
info = COMMUNES[["code", "nom", "codeEpci", "dep"]].rename(columns={"code": "commune"})
COM = agreger(P, ["dep", "commune"]).merge(info, on=["dep", "commune"], how="left")
# À la commune : uniquement des faits mesurés (prairies et enjeux), pas de risque prédit
COM = COM[[c for c in COM.columns if not c.startswith(("risque", "ha_top10", "ha_prio"))]]   # retire aussi ha_top10_*
COM.loc[COM.n_pp < MIN_PP_MAILLE, [c for c in COM.columns if c.startswith("ha_")]] = np.nan
COM.to_csv(SORTIES / "enjeux_communes_2025.csv", index=False)
EPCI = agreger(P.merge(info[["commune", "codeEpci"]], on="commune", how="left"), ["codeEpci"])
try:
    noms = pd.DataFrame(SESSION.get("https://geo.api.gouv.fr/epcis", params={"fields": "nom,code"}, timeout=120).json())
    EPCI = EPCI.merge(noms.rename(columns={"code": "codeEpci", "nom": "nom_epci"}), on="codeEpci", how="left")
except Exception as e:
    print("Noms des intercos non récupérés :", e)
EPCI.to_csv(SORTIES / "priorites_epci_2027.csv", index=False)
M = agreger(P, ["m10"])
M["x0_l93"] = M["m10"].str.split("_").str[0].astype(int) * 10_000      # coin sud-ouest en Lambert 93 (m)
M["y0_l93"] = M["m10"].str.split("_").str[1].astype(int) * 10_000
M = M.merge(P.assign(e=P.risque * P.ha).groupby("m10")["e"].sum().rename("ha_attendus_campagne_2027").round(1).reset_index(), on="m10")
M[M.n_pp >= MIN_PP_MAILLE].to_csv(SORTIES / "priorites_maille10km_2027.csv", index=False)
print("\n10 intercos avec le plus d'hectares prioritaires (niveau 1) :")
print(EPCI.sort_values("ha_prio1", ascending=False).head(10)[[c for c in ["nom_epci", "codeEpci", "ha_pp", "ha_prio1", "ha_prio2"] if c in EPCI]].to_string(index=False))

In [ ]:
# 5 bis. Pour classer les territoires, le modèle apporte-t-il plus que l'historique ?
# Test sur 2025 avec ce qui était connu fin 2024 : modèle entraîné sur a0 ≤ 2021, appliqué aux prairies 2023,
# contre la simple répétition du passé (taux de conversion observé 2019-2023 dans le même territoire).
# Règle écrite AVANT le résultat : à une échelle donnée, le modèle « apporte » si sa corrélation dépasse
# celle de l'historique d'au moins 0,05 ; sinon la carte affiche l'historique (plus simple) à cette échelle.
TOUT = pd.concat([TAB[TAB.a0 < 2022], NOUV], ignore_index=True)
TOUT = TOUT.merge(COMMUNES[["code", "codeEpci"]].rename(columns={"code": "commune"}), on="commune", how="left")
TOUT["m10"] = (TOUT.cx1 // 10).astype(int).astype(str) + "_" + (TOUT.cy1 // 10).astype(int).astype(str)
appr = TOUT[(TOUT.a0 <= 2021) & TOUT.y2.notna()]
cols2 = [v for v in VARS if appr[v].notna().any()]
mod2 = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000)).fit(appr[cols2], appr.y2)
test = TOUT[(TOUT.a0 == 2023) & TOUT.y2.notna()].copy()
test["s"] = mod2.predict_proba(test[cols2])[:, 1]
passe = TOUT[TOUT.a0 <= 2022]                                  # conversions 2019-20 … 2022-23, connues fin 2024
VA = []
for nom, cle, mini in [("commune", "commune", 50), ("maille 10 km", "m10", 200), ("interco", "codeEpci", 500)]:
    h = passe.assign(c=passe.y * passe.surface_ha).groupby(cle).agg(ha=("surface_ha", "sum"), c=("c", "sum"))
    t = test.assign(e=test.s * test.surface_ha, o=test.y2 * test.surface_ha).groupby(cle).agg(
        ha=("surface_ha", "sum"), e=("e", "sum"), o=("o", "sum"))
    t = t[t.ha >= mini].join((h.c / h.ha).rename("hist"), how="inner")
    t["prevu"], t["observe"] = t["e"] / t["ha"], t["o"] / t["ha"]       # noms de colonnes entre crochets (t.hist désigne une méthode pandas)
    rm = t["prevu"].corr(t["observe"], method="spearman"); rh = t["hist"].corr(t["observe"], method="spearman")
    rc = ((t["prevu"].rank() + t["hist"].rank()) / 2).corr(t["observe"], method="spearman")
    VA.append({"echelle": nom, "unites": len(t), "modele": round(rm, 2), "historique": round(rh, 2),
               "modele+historique": round(rc, 2), "le modele apporte": bool(rm >= rh + 0.05)})
VA = pd.DataFrame(VA)
print(VA.to_string(index=False))
VA.to_csv(SORTIES / "valeur_ajoutee_modele_vs_historique.csv", index=False)

In [ ]:
# 5 ter. Et à la parcelle ? Le modèle trouve-t-il plus de conversions que l'historique local ?
# Même test que 5 bis (modèle entraîné sur a0 ≤ 2021, prairies 2023, conversions réelles 2024→25),
# mais on mesure le RAPPEL : part des conversions réelles qui tombent dans les 10 % de prairies signalées.
# Concurrent : le taux de conversion 2019-2023 du secteur de la prairie (maille 1 km, 5 km, commune).
# Règle écrite AVANT le résultat : le modèle « apporte » à la parcelle si son rappel dépasse
# celui du meilleur historique d'au moins 3 points.
def rappel10(y, s):
    y, s = np.asarray(y, float), np.nan_to_num(np.asarray(s, float), nan=-1.0)   # secteur sans historique = en bas
    top = np.argsort(-s)[: int(0.10 * len(y))]
    return y[top].sum() / y.sum()

p = passe.assign(c=passe.y * passe.surface_ha)
t = test.copy()
for df in (p, t):
    df["m1"] = df.cx1.astype(str) + "_" + df.cy1.astype(str)                     # maille de 1 km
    df["m5"] = (df.cx1 // 5).astype(str) + "_" + (df.cy1 // 5).astype(str)       # maille de 5 km
res = {"modèle": rappel10(t.y2, t.s)}
for cle, nom in [("m1", "maille 1 km"), ("m5", "maille 5 km"), ("commune", "commune")]:
    g = p.groupby(cle).agg(c=("c", "sum"), ha=("surface_ha", "sum"))
    t[f"h_{cle}"] = t[cle].map(g["c"] / g["ha"])
    res[f"historique 2019-23, {nom}"] = rappel10(t.y2, t[f"h_{cle}"])
# Combinaison simple : moyenne des rangs du modèle et de l'historique à 1 km (pour information)
t["comb"] = (t.s.rank(pct=True) + t.h_m1.rank(pct=True).fillna(0)) / 2
res["combinaison modèle + historique 1 km"] = rappel10(t.y2, t.comb)
R = pd.DataFrame({"methode": list(res), "rappel_top10": [round(v, 3) for v in res.values()]})
print(R.to_string(index=False))
meilleur_hist = max(v for k, v in res.items() if k.startswith("historique"))
print(f"\nLe modèle apporte à la parcelle (règle +3 points) : {res['modèle'] >= meilleur_hist + 0.03}")
R.to_csv(SORTIES / "valeur_ajoutee_parcelle.csv", index=False)
print("Usage : ciblage à la parcelle pour un partenaire (parc, syndicat d'eau), jamais publié.")

In [ ]:
# 6. Valoriser : communes où les prairies se maintiennent (≥ 100 ha, autant ou plus de créations que de conversions présumées)
#    Prudence : un maintien peut venir du contexte (marais, pente) autant que des pratiques -> « maintien », pas « exemplarité »
O = pd.read_csv(SORTIES / "observatoire_communes_2019_2025.csv", dtype={"dep": str, "code": str, "codeEpci": str})
O = O.groupby(["dep", "code"], dropna=False)[["ha_retournes", "ha_crees"]].sum().reset_index().assign(nom=None)
pp_com = P.groupby(["dep", "commune"])["ha"].sum().rename("ha_pp_2025").reset_index().rename(columns={"commune": "code"})
X = pp_com.merge(O.drop(columns="nom"), on=["dep", "code"], how="left").fillna({"ha_retournes": 0, "ha_crees": 0})
X = X.merge(COMMUNES[["code", "nom"]], on="code", how="left")
X = X[X.ha_pp_2025 >= 100]                       # communes avec au moins 100 ha de prairies permanentes
X["taux_conversion_6ans"] = X.ha_retournes / X.ha_pp_2025
X["solde_ha"] = X.ha_crees - X.ha_retournes
X.to_csv(SORTIES / "communes_bilan_prairies_2019_2025.csv", index=False)
print(f"Communes avec ≥ 100 ha de prairies : {len(X)} ; taux de conversion médian sur 6 ans : {X.taux_conversion_6ans.median():.1%}")
print("\nCommunes où les prairies se maintiennent (solde positif ou nul, triées par surface de prairies) :")
print(X[X.solde_ha >= 0].sort_values("ha_pp_2025", ascending=False).head(15)[["dep", "nom", "ha_pp_2025", "ha_retournes", "ha_crees"]].to_string(index=False))

In [ ]:
# 7. Export pour la carte web : géométries simplifiées (WGS84) + indicateurs agrégés, dans un seul zip (carte_web.zip)
import shutil, shapely
EXP = SORTIES / "carte_web"; EXP.mkdir(exist_ok=True)
def ecrire(gdf, nom, tol):
    g = gdf.copy()
    g["geometry"] = g.geometry.simplify(tol, preserve_topology=True)
    g = g.to_crs(4326)
    g["geometry"] = shapely.set_precision(np.asarray(g.geometry.values, dtype=object), 1e-5)   # ~1 m : fichiers légers
    g.to_file(EXP / f"{nom}.geojson", driver="GeoJSON")
    print(f"   {nom}.geojson : {len(g):,} objets, {(EXP / f'{nom}.geojson').stat().st_size / 1e6:.1f} Mo")

# Observatoire : conversions et créations par commune et par année (colonnes larges)
obs = pd.read_csv(SORTIES / "observatoire_communes_2019_2025.csv", dtype={"dep": str, "code": str, "codeEpci": str})
obs["an"] = obs["transition"].str[-4:]
large = obs.pivot_table(index="code", columns="an", values=["ha_retournes", "ha_crees"], aggfunc="sum", fill_value=0)
large.columns = [f"{'conv' if a == 'ha_retournes' else 'crea'}_{b}" for a, b in large.columns]
large = large.round(1).reset_index().rename(columns={"code": "commune"})

# 1) Communes : faits mesurés uniquement (prairies 2025, enjeux, conversions/créations 2019-2025)
cg = COMMUNES[["code", "nom", "codeEpci", "dep", "geometry"]].rename(columns={"code": "commune"})
cg = cg.merge(COM.drop(columns=["nom", "codeEpci"], errors="ignore"), on=["dep", "commune"], how="left").merge(large, on="commune", how="left")
conv_cols = [c for c in cg.columns if c.startswith("conv_")]
# Pas de masquage des conversions mesurées : elles se déduisent du RPG public, et masquer les communes sans prairie
# en 2025 cacherait justement celles qui ont tout perdu. Aucune prédiction n'est exportée à la commune.
ecrire(cg, "communes", 80)

# 2) Intercos : hectares attendus (notebook 3), priorités et historique observé
eg = cg[["codeEpci", "geometry"]].dissolve(by="codeEpci").reset_index()
hist = (large.merge(COMMUNES[["code", "codeEpci"]].rename(columns={"code": "commune"}), on="commune")
             .drop(columns="commune").groupby("codeEpci").sum().round(1).reset_index())     # sommes non masquées
eg = eg.merge(hist, on="codeEpci", how="left").merge(EPCI, on="codeEpci", how="left")
v5 = SORTIES / "epci_hectares_attendus_2027.csv"
if v5.exists():
    att = pd.read_csv(v5, dtype={"codeEpci": str})[["codeEpci", "ha_attendus_campagne_2027", "ha_bas_90", "ha_haut_90"]]
    eg = eg.merge(att, on="codeEpci", how="left")
ecrire(eg, "intercos", 150)

# 3) Mailles de 10 km : carrés reconstruits depuis le coin sud-ouest
mg = M[M.n_pp >= MIN_PP_MAILLE].copy()
mg = gpd.GeoDataFrame(mg, geometry=[box(x, y, x + 10_000, y + 10_000) for x, y in zip(mg.x0_l93, mg.y0_l93)], crs=2154)
ecrire(mg, "mailles10km", 0)

# 4) Tableaux de synthèse + figures
for f in ["bilan_transitions_2019_2025.csv", "evaluation_complete.csv", "effet_regime_2023_2024.csv",
          "cultures_de_remplacement.csv", "qualite_etiquettes.csv", "communes_bilan_prairies_2019_2025.csv"]:
    if (SORTIES / f).exists():
        shutil.copy(SORTIES / f, EXP / f)
if (SORTIES / "figures").exists():
    shutil.copytree(SORTIES / "figures", EXP / "figures", dirs_exist_ok=True)
archive = shutil.make_archive(str(SORTIES / "carte_web"), "zip", EXP)
print(f"\nDonnées de la carte : {archive} ({Path(archive).stat().st_size / 1e6:.1f} Mo). Aucune donnée à la parcelle dedans.")

In [ ]:
# 8.
import re
PUB = Path("publiable")
if PUB.exists():
    shutil.rmtree(PUB)
(PUB / "donnees").mkdir(parents=True); (PUB / "figures").mkdir()
AUTORISES = ["surfaces_par_dep_annee_2015_2025.csv", "bilan_transitions_2019_2025.csv", "observatoire_communes_2019_2025.csv",
             "qualite_etiquettes.csv", "cultures_de_remplacement.csv", "concordance_retournement_declare_2024.csv",
             "resultats_modeles_normandie.csv", "verification_prediction_2025.csv", "evaluation_complete.csv",
             "effet_regime_2023_2024.csv", "epci_hectares_attendus_2027.csv", "importance_variables.csv", "sobriete_v5.csv",
             "versions_bibliotheques.txt", "DICTIONNAIRE_DES_DONNEES.md",
             "enjeux_communes_2025.csv", "priorites_epci_2027.csv", "priorites_maille10km_2027.csv", "priorites_synthese_normandie.csv",
             "valeur_ajoutee_modele_vs_historique.csv", "valeur_ajoutee_parcelle.csv", "communes_bilan_prairies_2019_2025.csv"]
absents = [f for f in AUTORISES if not (SORTIES / f).exists()]
for f in AUTORISES:
    if f not in absents:
        shutil.copy(SORTIES / f, PUB / "donnees" / f)
for g in ["communes", "intercos", "mailles10km"]:
    shutil.copy(EXP / f"{g}.geojson", PUB / "donnees" / f"{g}.geojson")
for f in sorted((SORTIES / "figures").glob("*.png")):
    shutil.copy(f, PUB / "figures" / f.name)

# Contrôle : aucun identifiant d'exploitant, aucun chemin, aucune colonne à la parcelle
INTERDIT = re.compile("|".join(["pac" + "age", r"\bi" + r"up\b", "num_" + "parcel", "num_" + "ilot", "/content/", "MyDrive"]), re.I)
COLONNES_PARCELLE = {"pid", "score", "top10", "p", "y", "y2", "cx1", "cy1"}
problemes = []
for f in sorted((PUB / "donnees").iterdir()):
    if INTERDIT.search(f.read_text(encoding="utf-8", errors="ignore")):
        problemes.append(f"{f.name} : identifiant ou chemin")
    if f.suffix == ".csv":
        cols = set(pd.read_csv(f, nrows=0).columns) & COLONNES_PARCELLE
        if cols:
            problemes.append(f"{f.name} : colonnes à la parcelle {sorted(cols)}")
print("Fichiers absents (notebook précédent non lancé ?) :", absents or "aucun")
print("Contrôle :", problemes or "OK, aucun identifiant, aucun chemin, aucune colonne à la parcelle")
archive_pub = shutil.make_archive("publiable", "zip", PUB)
